# 00 · Setup — Catalog / Schema / Volume / Synthetic Dataset

이 핸즈온 전체에서 사용할 인프라를 준비합니다.

| 리소스 | 용도 |
| --- | --- |
| Catalog `main` | UC 모델 / 테이블의 최상위 네임스페이스 |
| Schema `model_serving_cookbook` | 노트북 전체에서 사용 |
| Volume `artifacts` | wheel / joblib 등 binary artifact |
| Table `customers` | Synthetic customer churn 데이터셋 |

## 사전 요구사항
- Unity Catalog 활성화된 워크스페이스
- 본인이 `main` 카탈로그에 `USE CATALOG` + `CREATE SCHEMA` 권한 보유 (없으면 `config.py` 에서 본인 권한 있는 catalog로 변경)
- DBR ML 15.x 이상 권장 (MLflow 2.20+ 포함)

In [ ]:
%pip install -q "mlflow>=2.20.0" "databricks-sdk>=0.30.0"
%restart_python

In [ ]:
%run ./config

## Catalog / Schema / Volume 생성

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")
spark.sql(f"USE {catalog}.{schema}")
print(f"✓ {catalog}.{schema} ready (volume: {volume_path})")

## Synthetic Customer Churn 데이터 생성

핸즈온 전반에서 사용할 작은 데이터셋. Churn 이진분류용.

| 컬럼 | 타입 | 의미 |
| --- | --- | --- |
| `customer_id` | int | PK |
| `age` | int | 나이 |
| `tenure_months` | int | 가입 개월 수 |
| `monthly_charges` | float | 월 청구액 |
| `total_charges` | float | 누적 청구액 |
| `support_tickets` | int | 지난 30일 문의 건수 |
| `churned` | int | 0/1 — 타겟 |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

np.random.seed(42)
N = 5000

X, y = make_classification(
    n_samples=N,
    n_features=5,
    n_informative=4,
    n_redundant=0,
    n_classes=2,
    weights=[0.75, 0.25],
    random_state=42,
)

df = pd.DataFrame({
    "customer_id":      np.arange(1, N + 1),
    "age":              (X[:, 0] * 12 + 45).clip(18, 90).astype(int),
    "tenure_months":    (X[:, 1] * 18 + 30).clip(0, 120).astype(int),
    "monthly_charges":  (X[:, 2] * 25 + 75).clip(15, 200).round(2),
    "total_charges":    None,   # 아래에서 채움
    "support_tickets":  (X[:, 3] * 3 + 4).clip(0, 30).astype(int),
    "churned":          y.astype(int),
})
df["total_charges"] = (df["monthly_charges"] * df["tenure_months"]).round(2)

display(df.head())
print(f"shape={df.shape}, churn rate={df['churned'].mean():.2%}")

In [ ]:
(
    spark.createDataFrame(df)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.customers")
)
print(f"✓ {catalog}.{schema}.customers 저장 완료")

In [ ]:
display(spark.table(f"{catalog}.{schema}.customers").limit(5))

## MLflow 설정

- Registry 를 Unity Catalog 로 고정 (Workspace Model Registry 가 아님)
- Experiment 경로 지정

In [ ]:
import mlflow

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)
print(f"✓ MLflow registry = databricks-uc, experiment = {experiment_path}")

## 다음 단계

→ **`01_mlflow_basics`** 로 진행. Flavor 기반 logging + UC 등록 + alias 사용법.